# Clase 1 · Redes neuronales en la práctica

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hizocar/analitica-negocios-2/blob/main/docs/notebooks/clase1_redes_neuronales.ipynb)

**Analítica para los Negocios II · USM** · Notebook de la Clase 1 · Se aplica en el **Sprint 1 (domingo 8 de noviembre, 23:59)**.

En este notebook harás, con los datos **completos** de tu caso (~6.000 textos), el recorrido del Sprint 1:

1. Cargar los datos del caso (**FinTrust** o **MercaSur**).
2. **Auditar** su calidad y **limpiarlos** con una función reutilizable.
3. **EDA de texto:** clases, largo de los textos y palabras más frecuentes por clase.
4. Convertir el texto en números con **TF-IDF**.
5. Entrenar un **baseline**: regresión logística.
6. Entrenar una **red neuronal MLP en Keras** con *dropout* y *early stopping*, y leer sus curvas de entrenamiento.
7. **Comparar en test** con F1 macro, `classification_report` y matriz de confusión, y leer los errores con ojos de negocio.

> 💡 Primero haz **Archivo → Guardar una copia en Drive** para no perder tus cambios. No necesitas GPU ni clave de API.
> Todo el notebook corre en ~2 minutos con **Entorno de ejecución → Ejecutar todas**.

## 0. Configuración

Elige tu caso cambiando **una sola línea**. Todo el resto del notebook se adapta solo.

In [ ]:
CASO = "fintrust"  # o "mercasur"

BASE = "https://raw.githubusercontent.com/hizocar/analitica-negocios-2/main/docs/datasets/"

import unicodedata
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 120)
SEMILLA = 42

# Lo que cambia entre un caso y otro
CONFIG = {
    "fintrust": {
        "archivo": "reclamos_fintrust.csv",
        "texto": "texto",
        "clase": "categoria",
        "clases": ["Fraude", "Cobro indebido", "App y canales digitales",
                   "Crédito y tasas", "Atención al cliente", "Información y trámites"],
        "sinonimos": {"cobros indebidos": "Cobro indebido"},
        "clase_critica": "Fraude",   # el error más caro del negocio
    },
    "mercasur": {
        "archivo": "resenas_mercasur.csv",
        "texto": "texto",
        "clase": "aspecto",
        "clases": ["Despacho", "Calidad del producto", "Precio",
                   "Atención", "Devoluciones y garantía", "Pago"],
        "sinonimos": {"calidad producto": "Calidad del producto", "pagos": "Pago"},
        "clase_critica": "Pago",     # problemas de pago = ventas perdidas y riesgo de cobros erróneos
    },
}
cfg = CONFIG[CASO]
COL_TEXTO, COL_CLASE = cfg["texto"], cfg["clase"]
print(f"Caso: {CASO} · archivo: {cfg['archivo']} · columna a predecir: {COL_CLASE}")

## 1. Cargar los datos

Leemos el CSV directamente desde GitHub. Mira las primeras filas: ¿qué columnas hay?, ¿qué columna es el **texto** y cuál la **etiqueta** que queremos predecir?

In [ ]:
df_raw = pd.read_csv(BASE + cfg["archivo"])
print("Filas y columnas:", df_raw.shape)
df_raw.head()

## 2. Auditoría de calidad

Antes de modelar, **contamos los problemas**. Los datos vienen sucios a propósito (como los reales): duplicados, textos vacíos o basura, categorías mal escritas y valores imposibles.

La función `auditar` devuelve una tabla **problema → cuántos**, que puedes pegar tal cual en tu Sprint 1.

In [ ]:
def auditar(df):
    texto = df[COL_TEXTO].fillna("").astype(str).str.strip()
    filas = [
        ("Filas totales", len(df)),
        ("Duplicados exactos", int(df.duplicated().sum())),
        ("Texto nulo", int(df[COL_TEXTO].isna().sum())),
        ("Texto vacío o con menos de 3 palabras", int((texto.str.split().str.len() < 3).sum())),
        (f"Formas distintas de escribir '{COL_CLASE}' (deberían ser 6)", int(df[COL_CLASE].nunique())),
        (f"'{COL_CLASE}' nula", int(df[COL_CLASE].isna().sum())),
    ]
    # Valores imposibles propios de cada caso
    if CASO == "fintrust":
        filas += [
            ("Montos reclamados negativos", int((df["monto_reclamado"] < 0).sum())),
            ("Días de resolución < 0 o > 365", int(((df["dias_resolucion"] < 0) | (df["dias_resolucion"] > 365)).sum())),
            ("Urgencia nula", int(df["urgencia"].isna().sum())),
            ("Formas distintas de escribir 'canal'", int(df["canal"].nunique())),
        ]
    else:
        filas += [
            ("Rating nulo", int(df["rating"].isna().sum())),
            ("Rating fuera de 1–5", int((~df["rating"].isin([1, 2, 3, 4, 5]) & df["rating"].notna()).sum())),
            ("Precio <= 0", int((df["precio_clp"] <= 0).sum())),
        ]
    return pd.DataFrame(filas, columns=["problema", "cuántos"])

auditar(df_raw)

La herramienta estrella de la auditoría es `value_counts()`. Mira cuántas formas distintas hay de escribir la misma categoría:

In [ ]:
df_raw[COL_CLASE].value_counts()

## 3. Limpieza mínima (función reutilizable)

Decisiones de limpieza para el texto y la etiqueta:

| Problema | Decisión | Por qué |
|---|---|---|
| Duplicados exactos | Eliminar | Inflan el entrenamiento y pueden quedar a la vez en train y en test (el modelo "ya vio" la respuesta). |
| Texto vacío o basura ("reclamo", "ok", "...") | Eliminar | No hay nada que clasificar. |
| Categoría mal escrita ("FRAUDE", "fraude ", "Credito y tasas") | **Normalizar** y mapear | Es información válida, solo mal escrita: la recuperamos. |
| Valores imposibles (montos negativos, ratings fuera de rango) | Convertir en nulo | No los usamos en el modelo de texto, pero no deben contaminar otros análisis. |

La normalización deja todo en minúsculas, sin tildes y sin espacios extra, y luego un diccionario lleva cada variante a su nombre oficial.

In [ ]:
def sin_tildes(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

def clave(s):
    """'  Crédito y  TASAS ' -> 'credito y tasas'"""
    return " ".join(sin_tildes(str(s)).lower().split())

MAPA = {clave(c): c for c in cfg["clases"]}
MAPA.update(cfg["sinonimos"])

def limpiar(df):
    df = df.drop_duplicates().copy()
    df[COL_TEXTO] = df[COL_TEXTO].fillna("").astype(str).str.strip()
    df = df[df[COL_TEXTO].str.split().str.len() >= 3]
    df[COL_CLASE] = df[COL_CLASE].map(lambda s: MAPA.get(clave(s)) if pd.notna(s) else None)
    df = df.dropna(subset=[COL_CLASE])
    # Valores imposibles -> nulo
    if CASO == "fintrust":
        df.loc[df["monto_reclamado"] < 0, "monto_reclamado"] = np.nan
        df.loc[(df["dias_resolucion"] < 0) | (df["dias_resolucion"] > 365), "dias_resolucion"] = np.nan
    else:
        df.loc[~df["rating"].isin([1, 2, 3, 4, 5]), "rating"] = np.nan
        df.loc[df["precio_clp"] <= 0, "precio_clp"] = np.nan
    return df.reset_index(drop=True)

df = limpiar(df_raw)
print(f"Filas: {len(df_raw)} -> {len(df)} (eliminadas: {len(df_raw) - len(df)})")
print(f"Clases después de limpiar: {df[COL_CLASE].nunique()}")
df[COL_CLASE].value_counts()

✅ Ahora hay exactamente **6 clases**. Guarda esta comparación **antes / después** para tu informe.

## 4. EDA de texto

Tres preguntas: ¿las clases están **balanceadas**?, ¿qué tan **largos** son los textos?, ¿qué **palabras** caracterizan a cada clase?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 4.1 Distribución de clases
conteo = df[COL_CLASE].value_counts().sort_values()
axes[0].barh(conteo.index, conteo.values, color="#005f96")
axes[0].set_title("¿Cuántos textos hay de cada clase?")
axes[0].set_xlabel("n.º de textos")

# 4.2 Largo de los textos (en palabras)
df["n_palabras"] = df[COL_TEXTO].str.split().str.len()
axes[1].hist(df["n_palabras"], bins=30, color="#e5b300", edgecolor="white")
axes[1].axvline(df["n_palabras"].median(), ls="--", color="#b30828", label="mediana")
axes[1].set_title("¿Qué tan largos son los textos?")
axes[1].set_xlabel("n.º de palabras")
axes[1].legend()
plt.tight_layout()
plt.show()

# Largo por clase: ¿alguna clase escribe mucho más o mucho menos?
df.groupby(COL_CLASE)["n_palabras"].describe()[["count", "mean", "50%", "min", "max"]].round(1)

Para las palabras frecuentes conviene sacar las **stopwords**: palabras muy comunes en español ("el", "de", "que", "mi"…) que aparecen en todas las clases y no aportan. scikit-learn no trae una lista en español, así que usamos una corta hecha a mano (puedes ampliarla).

In [ ]:
STOPWORDS = set("""
a al algo ante como con contra cual cuando de del desde donde e el ella ellos en entre era es esa ese eso esta
este esto fue ha hace hay la las le les lo los me mi mis muy nada ni no nos o os para pero por porque que se
sin sobre su sus te ti tu un una uno unos y ya yo he han tengo tiene estoy está están son ser fue fui hoy
ahora gracias hola favor quiero necesito les cuento quedo atento atenta día días mes
""".split())

STOP = {sin_tildes(s) for s in STOPWORDS}

def tokens(texto):
    palabras = sin_tildes(texto.lower())
    palabras = "".join(c if c.isalnum() else " " for c in palabras).split()
    return [p for p in palabras if p not in STOP and len(p) > 2 and not p.isdigit()]

top = {}
for c in cfg["clases"]:
    cnt = Counter(t for texto in df.loc[df[COL_CLASE] == c, COL_TEXTO] for t in tokens(texto))
    top[c] = [f"{p} ({n})" for p, n in cnt.most_common(8)]
pd.DataFrame(top)

🔎 **Pregunta para tu informe:** ¿las palabras más frecuentes de cada clase son distintas entre sí? Si cada clase tiene su propio vocabulario, un modelo lineal sobre TF-IDF debería funcionar bien. ¿Hay alguna palabra que aparezca en varias clases y pueda confundir al modelo?

## 5. Separar entrenamiento y test

Separamos **antes** de cualquier transformación que "aprenda" de los datos. Usamos `stratify` para mantener la proporción de clases en ambos grupos. El test se mira **una sola vez**, al final.

In [ ]:
from sklearn.model_selection import train_test_split

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    df[COL_TEXTO], df[COL_CLASE], test_size=0.2, stratify=df[COL_CLASE], random_state=SEMILLA)
print(f"Entrenamiento: {len(X_train_txt)} textos · Test: {len(X_test_txt)} textos")

## 6. TF-IDF: el texto se convierte en números

Cada texto pasa a ser una fila con hasta 3.000 columnas (palabras y pares de palabras). Los parámetros:

- `max_features=3000`: nos quedamos con los 3.000 términos más frecuentes (liviano para la red).
- `ngram_range=(1, 2)`: palabras sueltas **y** pares ("no reconozco", "llegó tarde").
- `min_df=2`: ignorar términos que aparecen en un solo texto (suelen ser errores de tipeo).
- `sublinear_tf=True`: usa log(1 + conteo), para que repetir una palabra 5 veces no pese 5 veces más.

⚠️ `fit_transform` **solo** en entrenamiento; en test solo `transform`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train = vec.fit_transform(X_train_txt)
X_test = vec.transform(X_test_txt)
print("Matriz de entrenamiento:", X_train.shape, f"· {X_train.nnz / np.prod(X_train.shape):.1%} de celdas distintas de cero")

## 7. Baseline: TF-IDF + regresión logística

Siempre partimos con un modelo simple. Es la vara que la red neuronal tiene que superar para justificar su complejidad.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay

logit = LogisticRegression(max_iter=2000, C=5)
logit.fit(X_train, y_train)
pred_logit = logit.predict(X_test)
f1_logit = f1_score(y_test, pred_logit, average="macro")
print(f"Regresión logística · F1 macro en test = {f1_logit:.3f}")

## 8. Red neuronal MLP en Keras

Arquitectura:

```
TF-IDF (3.000) → Dense(128, ReLU) → Dropout(30 %) → Dense(6, softmax)
```

- **Dense(128, ReLU):** capa oculta con 128 neuronas.
- **Dropout(0.3):** apaga al azar el 30 % de esas neuronas en cada paso de entrenamiento (contra el sobreajuste).
- **Dense(6, softmax):** una salida por clase; softmax entrega 6 probabilidades que suman 1.
- **Pérdida:** `sparse_categorical_crossentropy` (entropía cruzada con etiquetas como números 0–5).
- **Optimizador:** Adam (descenso de gradiente con tamaño de paso adaptativo).

Keras necesita **números** como etiquetas y **matrices densas** como entrada (TF-IDF es dispersa: la convertimos con `.toarray()`; con 3.000 columnas cabe sin problema en memoria).

In [ ]:
import keras
from keras import layers
from sklearn.preprocessing import LabelEncoder

keras.utils.set_random_seed(SEMILLA)  # resultados reproducibles

codificador = LabelEncoder().fit(y_train)
y_train_num = codificador.transform(y_train)
y_test_num = codificador.transform(y_test)
CLASES = list(codificador.classes_)

X_train_d = X_train.toarray().astype("float32")
X_test_d = X_test.toarray().astype("float32")

red = keras.Sequential([
    keras.Input(shape=(X_train_d.shape[1],)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(CLASES), activation="softmax"),
])
red.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])
red.summary()

### Entrenamiento con early stopping

- `validation_split=0.15`: Keras separa el **último** 15 % del entrenamiento como validación (nuestros datos ya vienen mezclados por `train_test_split`, así que está bien).
- `EarlyStopping(patience=3, restore_best_weights=True)`: si la pérdida de validación no mejora en 3 épocas seguidas, para y **recupera los pesos de la mejor época**.
- `batch_size=64`: cada paso de descenso de gradiente usa 64 textos.

In [ ]:
parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

historia = red.fit(
    X_train_d, y_train_num,
    validation_split=0.15,
    epochs=40,
    batch_size=64,
    callbacks=[parada],
    verbose=2,
)
mejor_epoca = int(np.argmin(historia.history["val_loss"])) + 1
print(f"Épocas entrenadas: {len(historia.history['loss'])} · mejor época (según validación): {mejor_epoca}")

### Curvas de entrenamiento

Léelas como en el simulador del Tema 4: si la pérdida de **entrenamiento** sigue bajando mientras la de **validación** sube, la red está memorizando. La línea roja marca la época cuyos pesos nos quedamos.

In [ ]:
h = historia.history
epocas = range(1, len(h["loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epocas, h["loss"], "o-", color="#e5b300", label="entrenamiento")
axes[0].plot(epocas, h["val_loss"], "o-", color="#005f96", label="validación")
axes[0].set_title("Pérdida (entropía cruzada)")
axes[1].plot(epocas, h["accuracy"], "o-", color="#e5b300", label="entrenamiento")
axes[1].plot(epocas, h["val_accuracy"], "o-", color="#005f96", label="validación")
axes[1].set_title("Exactitud")
for ax in axes:
    ax.axvline(mejor_epoca, ls="--", color="#b30828", label="mejor época")
    ax.set_xlabel("época")
    ax.legend()
plt.tight_layout()
plt.show()

## 9. Comparación en test

Ahora sí miramos el test, **una sola vez**, con los dos modelos ya decididos.

In [ ]:
prob_red = red.predict(X_test_d, verbose=0)
pred_red = codificador.inverse_transform(prob_red.argmax(axis=1))
f1_red = f1_score(y_test, pred_red, average="macro")

comparacion = pd.DataFrame({
    "modelo": ["TF-IDF + regresión logística", "TF-IDF + red MLP (Keras)"],
    "F1 macro (test)": [f1_logit, f1_red],
    "exactitud (test)": [(pred_logit == y_test).mean(), (pred_red == y_test.to_numpy()).mean()],
}).round(3)
comparacion

El F1 macro resume, pero no dice **dónde** se equivoca cada modelo. El `classification_report` muestra precisión, recall y F1 por clase:

In [ ]:
mejor_nombre, mejor_pred = (("Regresión logística", pred_logit) if f1_logit >= f1_red
                            else ("Red MLP (Keras)", pred_red))
print(f"Reporte por clase del mejor modelo: {mejor_nombre}\n")
print(classification_report(y_test, mejor_pred, digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (nombre, pred) in zip(axes, [("Regresión logística", pred_logit), ("Red MLP (Keras)", pred_red)]):
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=cfg["clases"], ax=ax,
                                            xticks_rotation=45, cmap="Blues", colorbar=False)
    ax.set_title(f"{nombre} · F1 macro = {f1_score(y_test, pred, average='macro'):.3f}")
    ax.set_xlabel("Clase predicha")
    ax.set_ylabel("Clase real")
plt.tight_layout()
plt.show()

## 10. Los errores con ojos de negocio

No todos los errores cuestan lo mismo. Miramos la **clase crítica** de tu caso (en FinTrust, *Fraude*; en MercaSur elegimos *Pago*, pero **tú decides** cuál es la crítica en tu proyecto y lo justificas): ¿cuántos casos reales de esa clase se escaparon y hacia dónde fueron?

In [ ]:
CRITICA = cfg["clase_critica"]
for nombre, pred in [("Regresión logística", pred_logit), ("Red MLP (Keras)", np.asarray(pred_red))]:
    reales = (y_test == CRITICA).to_numpy()
    escapados = pd.Series(np.asarray(pred)[reales & (np.asarray(pred) != CRITICA)])
    print(f"{nombre}: {reales.sum() - len(escapados)} de {reales.sum()} casos de '{CRITICA}' detectados "
          f"(recall = {1 - len(escapados) / reales.sum():.1%})")
    if len(escapados):
        print("   se fueron a:", {k: int(v) for k, v in escapados.value_counts().items()})

# Algunos textos de la clase crítica que el mejor modelo no detectó: ¿error del modelo o etiqueta mal puesta?
mask = (y_test == CRITICA).to_numpy() & (np.asarray(mejor_pred) != CRITICA)
pd.DataFrame({"texto": X_test_txt[mask], "real": y_test[mask], "predicho": np.asarray(mejor_pred)[mask]}).head(8)

🔎 Lee los textos de arriba. Muchos "errores" son en realidad **etiquetas mal puestas** (los datos traen ~6 % de ruido de etiqueta, como pasa en la vida real). Ningún modelo arregla eso: el techo del F1 lo pone la calidad de las etiquetas. Es un hallazgo que vale la pena reportar.

## 11. Ejercicios para tu proyecto (Sprint 1)

Usa este notebook como base. Para el buzón del **domingo 8 de noviembre, 23:59**:

1. **Pregunta de negocio.** Escribe en una celda de texto, en 2–3 líneas, qué decisión apoyará tu clasificador y quién lo usará.
2. **Auditoría completa.** Amplía `auditar` con al menos **dos** chequeos más (p. ej. variantes de `canal`, fechas fuera de rango, textos con datos personales como correos o RUT). Presenta una tabla resumen problema → decisión → filas afectadas.
3. **EDA con conclusiones.** Bajo cada gráfico del punto 4, escribe una conclusión de una línea. ¿Hay clases desbalanceadas? ¿Textos muy cortos? ¿Palabras que se repiten entre clases?
4. **Experimenta con la red.** Prueba al menos **dos** variantes (p. ej. 64 vs. 256 neuronas, dropout 0,2 vs. 0,5, o una segunda capa oculta) y registra en una tabla el F1 macro **de validación** de cada una (separa tú un grupo de validación desde `X_train` con `train_test_split`, o compara al menos la `val_loss` de `historia`). Elige con validación, no con test.
5. **Curvas.** Incluye las curvas de entrenamiento de tu red final y explica si hay sobreajuste y en qué época se detuvo.
6. **Comparación en test.** Tabla con F1 macro de la logística y de tu mejor red, `classification_report` y matriz de confusión de **ambos** modelos.
7. **Lectura de negocio (3–5 líneas).** ¿Cuál es el error más caro en tu caso? ¿Qué modelo recomiendas por ahora y por qué (calidad, simplicidad, explicabilidad)?

**Cómo entregar:** en la **primera celda** escribe los nombres del equipo, el caso y el enlace de Colab (con permiso de lectura). Luego *Entorno de ejecución → Reiniciar y ejecutar todo*, descarga el notebook (*Archivo → Descargar → Descargar .ipynb*), nómbralo `S1_FinTrust_Apellido1_Apellido2_Apellido3.ipynb` (o `S1_MercaSur_…`) y súbelo al [buzón del Sprint 1](https://hizocar.github.io/analitica-negocios-2/proyecto/buzon/). El buzón acepta dos días más con −1,0 punto por día de atraso.

> 💡 **Para la tutoría del jueves 5 de noviembre:** trae este notebook corriendo con tu caso y una pregunta concreta para tu equipo docente.